# From Failure to Fix: MAST classification — project Steps 1–6

This notebook contains the six project steps through classifier implementation. Step 6 contains its eight technical substeps, numbered 6.1–6.8. Historical experiment and recovery notebooks are archived in `backups`; they are not additional project tasks.

**Run from the beginning:** select the project `.venv` Python kernel, restart it, and Run All with `RUN_NEW_GEMINI_BATCH=False`. Public datasets download only when missing. Local checks and saved results are reproduced without new Gemini classifications. To deliberately classify new traces, use the single bounded-batch control under **6.8**, only when no other evaluation is active. Never share `.env`.

**Status is evidence-based:** successful code execution is not proof every model label is correct. The sample classifier has live pilot results with failures preserved. Full-corpus evaluation is a separate extension and remains incomplete while traces are unattempted.

Team handoff: [Teammate guide: Steps 1–6 and safe continuation](TEAMMATE_GUIDE_STEPS_1_TO_6.md).


## Setup — verified public dataset download

The first code cell downloads the pinned public full and human-labelled datasets only if missing (about 202 MB total), checks their hashes, and restores required local manifests. No Gemini key is needed for setup. Live classification uses Gemini, not the legacy Groq provider. A working Python kernel is required; `python-dotenv` is needed for live API execution.


In [1]:
# FIRST CODE CELL: download pinned public datasets only when missing.
# Fresh machine: approximately 202 MB. No Gemini key or pip packages required here.
from pathlib import Path
import importlib
import gemini_workflow as workflows
importlib.reload(workflows)
ROOT = Path.cwd()
if not (ROOT / "gemini_workflow.py").exists():
    raise RuntimeError("Open the notebook from the project folder containing gemini_workflow.py.")
DOWNLOAD_DATASETS = True  # Public Hugging Face download only; NOT Gemini execution.
ROOT = workflows.setup_project(ROOT, download=DOWNLOAD_DATASETS)
workflows.set_api_enabled(False)


Verified existing dataset: MAD_human_labelled_dataset.json
Verified existing dataset: MAD_full_dataset.json
Setup complete. Public data and frozen support files verified. No Gemini requests.
Saved Gemini results are optional local artifacts; no saved results are downloaded or recreated.


## Step 1 — Choose the MAST failure-category scope

The project initially considered a smaller category focus; the user later requested all 14 for an exploratory classifier. This does not establish which categories are most common or select the final 2–3 remediation targets. The four-category baseline is preserved historically.


In [2]:
import json, hashlib
workflow_v3 = workflows.get_mode("evidence_v3")
human_mode = workflows.get_mode("fourteen_category")
print("Current classifier: 14-category exploratory Gemini judge")
print("Categories:")
for category in workflow_v3.DEFINITIONS:
    print(" -", category)
print("Final remediation focus is a later team decision; predicted frequency is not ground truth.")


Current classifier: 14-category exploratory Gemini judge
Categories:
 - disobey_task_specification
 - disobey_role_specification
 - step_repetition
 - loss_of_conversation_history
 - unaware_of_termination_conditions
 - conversation_reset
 - fail_to_ask_for_clarification
 - task_derailment
 - information_withholding
 - ignored_other_agents_input
 - reasoning_action_mismatch
 - premature_termination
 - no_or_incomplete_verification
 - incorrect_verification
Final remediation focus is a later team decision; predicted frequency is not ground truth.


## Step 2 — Review the paper and taxonomy definitions

Use the MAST paper and source definitions to keep meaning consistent. Definitions are displayed below; source category headings/codes and annotation rounds are retained separately. Numeric codes changed meanings between revisions and are not universal identifiers. Displaying definitions does not certify that every teammate has read the paper or agreed on mappings. Paper: https://arxiv.org/abs/2503.13657


In [3]:
for category, definition in workflow_v3.DEFINITIONS.items():
    print(category + ": " + definition)
print("Source definitions are preserved; category equivalence is checked by definition body, not numeric code alone.")


disobey_task_specification: Failure to adhere to the specified constraints or requirements of a given task, leading to suboptimal or incorrect outcomes.
disobey_role_specification: Failure to adhere to the defined responsibilities and constraints of an assigned role, potentially leading to an agent behaving like another.
step_repetition: Unnecessary reiteration of previously completed steps in a process, potentially causing delays or errors in task completion.
loss_of_conversation_history: Unexpected context truncation, disregarding recent interaction history and reverting to an antecedent conversational state.
unaware_of_termination_conditions: Lack of recognition or understanding of the criteria that should trigger the termination of the agents’ interaction, potentially leading to unnecessary continuation.
conversation_reset: Unexpected or unwarranted restarting of a dialogue, potentially losing context and progress made in the interaction.
fail_to_ask_for_clarification: (between age

## Step 3 — Download and verify the MAST datasets

The setup cell performs the actual pinned download. The large corpus has AI-generated reference labels; the separate smaller dataset contains the three human annotators. Do not call the large corpus human-labelled ground truth.


In [4]:
REVISION = "95118ac951421753cf1deb87ddea3b01e693c41b"
RAW_DIR = ROOT / "data/raw" / ("mast_" + REVISION)
split_manifest = json.loads((ROOT / "data/processed/full_corpus_step6_95118ac95142/split_manifest.json").read_text(encoding="utf-8"))
human_records = json.loads((RAW_DIR / "MAD_human_labelled_dataset.json").read_text(encoding="utf-8"), parse_constant=lambda value: None)
print("Pinned upstream revision:", REVISION)
print("Full dataset bytes:", (RAW_DIR / "MAD_full_dataset.json").stat().st_size)
print("Full-corpus source records:", len({i for row in split_manifest["traces"] for i in row["source_record_indices"]}))
print("Unique full-corpus traces:", len(split_manifest["traces"]))
print("Human dataset records:", len(human_records))
print("Raw data hashes were verified by setup; source files are not edited.")


Pinned upstream revision: 95118ac951421753cf1deb87ddea3b01e693c41b
Full dataset bytes: 199574367
Full-corpus source records: 1642
Unique full-corpus traces: 1537
Human dataset records: 19
Raw data hashes were verified by setup; source files are not edited.


## Step 4 — Define the common trace-logging template

Use stable trace identity and preserve provenance, model output, review notes and uncertainty. `label_correct` stays unknown until an eligible human comparison/review; it is not set from an AI success claim. API records and source annotations already supply the underlying fields. The joined comparison and accuracy report belong to Step 7.


In [5]:
logging_template = {
    "trace_hash": None, "source_trace_id": None, "framework": None,
    "source_revision": REVISION, "annotation_round": None,
    "source_category_heading": None, "canonical_category": None,
    "human_votes": None, "majority_label": None, "model_prediction": None,
    "record_status": None, "label_correct": None, "review_notes": None,
}
print(json.dumps(logging_template, indent=2))


{
  "trace_hash": null,
  "source_trace_id": null,
  "framework": null,
  "source_revision": "95118ac951421753cf1deb87ddea3b01e693c41b",
  "annotation_round": null,
  "source_category_heading": null,
  "canonical_category": null,
  "human_votes": null,
  "majority_label": null,
  "model_prediction": null,
  "record_status": null,
  "label_correct": null,
  "review_notes": null
}


## Step 5 — Parse human annotations and calculate majority labels

Majority requires two agreeing Boolean votes. Missing/non-finite votes are missing, not False. Keep source schemes and original definitions; exact duplicate/conflicting references and unresolved category mappings are not silently combined. This cell shows the complete source-row count and the separately eligible mapped human references. It does not replace AI references with human labels.


In [6]:
source_ground_truth = []
for index, record in enumerate(human_records):
    for annotation in record["annotations"]:
        votes = [annotation.get("annotator_" + str(i)) for i in (1, 2, 3)]
        yes = sum(v is True for v in votes)
        no = sum(v is False for v in votes)
        majority = True if yes >= 2 else False if no >= 2 else None
        source_ground_truth.append({"record_index": index, "round": record.get("round"), "trace_hash": hashlib.sha256(record["trace"].encode("utf-8")).hexdigest(), "category_heading": annotation["failure mode"].splitlines()[0], "votes": [v if type(v) is bool else None for v in votes], "majority_label": majority})
human_data = human_mode.prepare(ROOT)
print("All original trace/category annotation rows:", len(source_ground_truth))
print("Mapped annotation rows retained:", len(human_data["annotations"]))
print("Eligible canonical human references:", len(human_data["references"]))
print("Conflicting annotation rows outside scoring:", len(human_data["conflicts"]))
print("Distinct unresolved source headings:", len(human_data["unmapped_definitions"]))
print("Example majority rows (no trace text):")
for row in source_ground_truth[:3]:
    print(json.dumps(row))


All original trace/category annotation rows: 316
Mapped annotation rows retained: 206
Eligible canonical human references: 190
Conflicting annotation rows outside scoring: 12
Distinct unresolved source headings: 15
Example majority rows (no trace text):
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.1 Poor task constraint compliance", "votes": [false, false, false], "majority_label": false}
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.2 Inconsistency between reasoning and action", "votes": [false, false, false], "majority_label": false}
{"record_index": 0, "round": "Round 1", "trace_hash": "a6abc5b76eed3380e2c2bb906437e47a97ee6494edd8d6942e9ac7afdc5e8abf", "category_heading": "1.3 Undetected conversation ambiguities and contradictions", "votes": [false, false, false], "majority_label": false

## Step 6 — Build an LLM-based classifier

**Deliverable:** versioned Gemini classifier, prompt/settings, strict parser tests and saved predictions with visible failure statuses. The eight substeps below implement this workflow. A live pilot exists; its model errors and scoped-review limitations remain documented. Full-dataset evaluation is optional to the original sample-classifier goal, but is being tracked separately because the user requested broad coverage. It is not complete.


### 6.1 Confirm scope and taxonomy

Use the selected canonical definitions while preserving original source headings/codes separately. The taxonomy is not changed to make predictions agree with references.


In [7]:
assert len(workflow_v3.DEFINITIONS) == 14
print("Scope: 14 categories; provider Gemini; model gemini-3.5-flash-lite")
print("Source annotation schemes are preserved. Unresolved mappings stay outside primary comparisons.")


Scope: 14 categories; provider Gemini; model gemini-3.5-flash-lite
Source annotation schemes are preserved. Unresolved mappings stay outside primary comparisons.


### 6.2 Create development and held-out samples

Prompt revisions use development only. Exact trace duplicates and conservative task components cannot cross the split. Near-duplicate absence is not proven; earlier human-evaluated traces remain exploratory.


In [8]:
rows = split_manifest["traces"]
dev_hashes = {r["trace_hash"] for r in rows if r["split"] == "development"}
test_hashes = {r["trace_hash"] for r in rows if r["split"] == "heldout"}
assert not dev_hashes & test_hashes
assert not {r["group_id"] for r in rows if r["split"] == "development"} & {r["group_id"] for r in rows if r["split"] == "heldout"}
print("Development:", len(dev_hashes), "| Held-out:", len(test_hashes))
print("Exact trace overlap: 0 | conservative component overlap: 0")


Development: 1214 | Held-out: 323
Exact trace overlap: 0 | conservative component overlap: 0


### 6.3 Repair and version the prompt

Evidence-v3 contains explicit definitions, label meanings, evidence requirements and a trace boundary. Recorded instructions are data, not commands. No malformed numeric-code example is used. The prompt and classifier configuration remain frozen during broad evaluation; no held-out tuning is allowed.


In [9]:
prompt_check = workflow_v3.prompt_for("SYNTHETIC FULL TRACE")
assert "BEGIN RECORDED TRACE" in prompt_check and "END RECORDED TRACE" in prompt_check
assert "SYNTHETIC FULL TRACE" in prompt_check
print("Prompt version: full-corpus-fourteen-evidence-v3")
print("Frozen classifier hash:", workflow_v3.RECOVERY_CODE_SHA256)
print("Trace-boundary and full-input prompt checks passed.")


Prompt version: full-corpus-fourteen-evidence-v3
Frozen classifier hash: 0b2f1743d816f8da8a134e7dffbe1d94220e8fe922b5fe963ea3653b657b11ab
Trace-boundary and full-input prompt checks passed.


### 6.4 Preserve input fidelity

Live Gemini countTokens checks the full prompt before generation. Oversized inputs are blocked intact, not trimmed or silently dropped. Character-based estimates are not exact provider token counts.


In [10]:
progress = workflows.evaluation_progress(ROOT)
fidelity_checked = 0
for item in progress["folders"]:
    for path in (ROOT / item["folder"]).glob("*.json"):
        if len(path.stem) != 64:
            continue
        record = json.loads(path.read_text(encoding="utf-8"))
        assert record["trace_was_trimmed"] is False
        assert record["original_characters"] == record["sent_characters"]
        assert isinstance(record["counted_input_tokens"], int)
        fidelity_checked += 1
print("Saved full-input/token-count records checked:", fidelity_checked)
print("No new API calls were made by this check.")


Saved full-input/token-count records checked: 105
No new API calls were made by this check.


### 6.5 Request structured output

V3 documents a three-way adaptation: present/absent/insufficient_evidence, parsed to Boolean True/False/None. None is not a negative label. Each canonical category requires an explanation and quotes; positive findings require literal evidence. Existence of a quote does not prove the interpretation is correct.


In [11]:
schema = workflow_v3.schema()
assert len(schema["required"]) == 14
print("14 required category identifiers; fields: decision, evidence, quotes")
print("Stored labels: present=True; absent=False; insufficient_evidence=None")
print("Invalid or missing responses are unavailable, never automatically False.")


14 required category identifiers; fields: decision, evidence, quotes
Stored labels: present=True; absent=False; insufficient_evidence=None
Invalid or missing responses are unavailable, never automatically False.


### 6.6 Replace and test the permissive parser

Strict checks reject missing/extra/duplicate categories, duplicate JSON fields, wrong decision types, unsupported positives and non-verbatim quotes. Numeric identifiers are not mapped without their source meaning. Historical free-form 2.5/3.3 outputs were rejected as incompatible, not silently relabelled. Run the full tests with the project .venv.


In [12]:
parser_checks = workflows.parser_contract_checks()
print("Valid complete answer accepted:", parser_checks["valid_full_answer_passed"])
print("Invalid parser cases rejected:", len(parser_checks["invalid_cases_rejected"]))
print("Abstentions preserved:", parser_checks["abstention_preserved_not_false"])
replay = ROOT / "data/processed/step6_consolidated_audit/parser_replay_report.json"
if replay.exists():
    replay_report = json.loads(replay.read_text(encoding="utf-8"))
    print("Actual archived coded answers rejected:", len(replay_report["historical_coded_responses"]))
    print("Canonical baseline responses reparsed identically:", len(replay_report["canonical_saved_responses"]))


Valid complete answer accepted: True
Invalid parser cases rejected: 10
Abstentions preserved: True
Actual archived coded answers rejected: 3
Canonical baseline responses reparsed identically: 36


### 6.7 Save resumable records

Records/manifests retain trace and prompt hashes, data revision/source scheme, provider/model/settings, full-input status, raw response/usage when returned, predictions and errors. Prior experiments remain intact. Any attempted v3 hash, including errors or unknown outcomes, is excluded from automatic resend.


In [13]:
progress = workflows.evaluation_progress(ROOT)
print("Preserved v3 experiment folders:", len(progress["folders"]))
print("Saved trace records:", sum(x["records"] for x in progress["folders"]))
print("No automatic retry of valid, invalid, failed, blocked or unknown-outcome trace records.")


Preserved v3 experiment folders: 3
Saved trace records: 105
No automatic retry of valid, invalid, failed, blocked or unknown-outcome trace records.


### 6.8 Pilot, freeze, and run a bounded Gemini batch

The 14-case online development pilot has 12 valid responses, one invalid quote response and one request failure without a saved response. Scoped AI-assisted evidence review and tests are documented; no exhaustive human correctness sign-off is fabricated. Known model errors are retained. Full held-out execution is blocked until development attempts are accounted for.

**Default:** RUN_NEW_GEMINI_BATCH=False displays current saved results only. **Live use:** set it True to attempt at most 20 untouched development traces / one million counted input tokens, using your .env key. Do not launch it while another run lock exists. HTTP 503 is a provider error; the batch stops, preserves that request, and does not retry it. The summary below refreshes AFTER a live batch, so it includes that batch rather than old pre-run totals.


In [14]:
RUN_NEW_GEMINI_BATCH = False
BATCH_SPLIT = "development"
BATCH_MAX_GENERATIONS = 20
BATCH_MAX_INPUT_TOKENS = 1_000_000
batch_result = None
if RUN_NEW_GEMINI_BATCH:
    locks = [ROOT / "saved_results/STEP6_V3_FULL_RUNNING.lock", ROOT / "saved_results/STEP6_V3_SUPERVISOR_RUNNING.lock"]
    if any(lock.exists() for lock in locks):
        raise RuntimeError("Another evaluation may be active. Do not start a competing batch.")
    print("Starting ONE bounded Gemini batch. This uses API quota; please wait for the refreshed summary.")
    workflows.set_api_enabled(True)
    try:
        batch_result = workflows.run_step6_batch(ROOT, split=BATCH_SPLIT, max_generations=BATCH_MAX_GENERATIONS, max_input_tokens=BATCH_MAX_INPUT_TOKENS, allow_api=True, verbose=False)
    finally:
        workflows.set_api_enabled(False)
else:
    print("Read-only mode: this cell is NOT sending new Gemini requests.")
progress = workflows.print_evaluation_summary(ROOT, batch_result=batch_result)
print("Sample pilot: 14 attempted; 12 valid; 2 invalid/missing. These are part of the totals above.")


Read-only mode: this cell is NOT sending new Gemini requests.
CURRENT SAVED RESULTS (refreshed now)
Development: 1214 total | 76 valid | 1109 unattempted
  Other outcomes (not negative labels): blocked_input_limit=1, error=2, generation_started=1, invalid_response=23, request_error=2
Heldout: 323 total | 0 valid | 323 unattempted
Latest saved response/record: 2026-10-19T16:08:11.052779+00:00
No evaluation run lock is present.
Latest batch: stopped_api_or_transport_error
Batch checkpoint time: 2026-10-19T16:08:11.057879+00:00
  Generation attempts: 16 | valid: 11 | invalid: 4 | blocked: 0 | counted generation input tokens: 405,843
  Stop reason: Gemini HTTP 503; no retry was attempted
  The interrupted trace was preserved and will not be automatically resent.
  Evaluation stopped; it is not currently continuing in this notebook.
Full-dataset coverage complete: False
Validation is not correctness. Invalid, blocked and unknown outcomes are not scored as absent.
Sample pilot: 14 attempted;

## Status and handoff to Step 7

The classifier, parser, versioned settings and sample predictions exist with documented errors. Invalid outputs are not repaired into successful labels. The full-dataset extension is **incomplete** while either split has unattempted traces; provider errors and blocks must stay in its final coverage report. Human manual pilot sign-off has not been recorded.

Step 7 is a separate comparison against eligible human labels and disagreement analysis. Agreement with upstream AI-reference labels is not human accuracy. Saved outputs and successful tests do not establish broad reliability.
